# 03a - RM-a: full fine-tuning IndoBERT

Baseline penelitian. Seluruh 109 juta parameter IndoBERT diperbarui, sehingga
skenario ini yang paling mahal sekaligus menjadi pembanding performa untuk dua
strategi ringan.

Notebook ini menjalankan SATU konfigurasi: baseline kanonik IndoNLU/Wilie (2020)
`lr=2e-5, epochs=5, batch=16, warmup=0,1, wd=0,01`. Eksplorasi hyperparameter
ada di `04_tuning_campaign.ipynb`.

Prasyarat: `02_preprocessing.ipynb` sudah dijalankan.

In [ ]:
from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.output_dir / "baseline"

runner = CampaignRunner(out_dir=OUT_DIR)
print("device        :", runner.device)
print("encoder       :", runner.model_name)
print("keluaran      :", runner.out_dir)
print("train/val/test:", [len(frame) for frame in runner.data.frames.values()])

## 1. Catat konteks hardware

In [ ]:
hardware = runner.write_hardware()
for key, value in hardware.items():
    print(f"  {key:16s}: {value}")

Angka efisiensi hanya bisa ditafsirkan bersama konteks ini, dan hanya sebanding
bila seluruh skenario diukur pada hardware dan sesi yang sama.

## 2. Konfigurasi

In [ ]:
from src.models.schemas import RMAConfig

config = RMAConfig()
print(config.model_dump())
print(f"\nbatch efektif {config.batch} dicapai lewat micro-batch "
      f"{config.effective_micro_batch} x akumulasi {config.grad_accum}")

Akumulasi gradien ekuivalen secara matematis dengan batch besar untuk BERT
(LayerNorm, bukan BatchNorm) selama loss dibagi jumlah akumulasi. Ini kompromi
memori, bukan kompromi hasil, dan yang membuat `batch=32` tetap bisa dijalankan
pada GPU 4 GB.

## 3. Jalankan

In [ ]:
row = runner.run(
    "rma",
    config.model_dump(),
    note="baseline kanonik IndoNLU/Wilie 2020, titik acuan seluruh grid",
)

print(f"run #{row['run_id']}")
print(f"  val F1-macro    : {row['val_f1_macro']:.4f}")
print(f"  val F1 judi     : {row['val_f1_judi']:.4f}")
print(f"  epoch terbaik   : {row['best_epoch']} dari {row['epochs']}")
print(f"  waktu latih     : {row['train_time_s']:.1f} s")
print(f"  peak GPU memory : {row['peak_mem_mb']:.0f} MB")
print(f"  trainable params: {row['trainable_params']:,}")

## 4. Kurva per epoch

In [ ]:
import pandas as pd

history = pd.read_csv(OUT_DIR / "history" / "rma_history.csv")
history[history["run_id"] == row["run_id"]]

`overfit_signal` di baris riwayat bernilai True bila epoch terbaik bukan epoch
terakhir, yaitu tanda bahwa menambah epoch justru memperburuk validasi.

Figur kurva tersimpan di `outputs/baseline/figures/rma_run{id}_curve.png`.

## Ringkasan

Angka di atas adalah baseline satu konfigurasi, bukan hasil final. Konfigurasi
final ditentukan lewat kampanye di `04_tuning_campaign.ipynb`, dan angka test
baru dibuka sekali di `05_final_benchmark.ipynb`.

Lanjut ke `03b_rmb_frozen.ipynb`.